In [ ]:
!wget http://content.udacity-data.com/courses/ud617/purchases.txt.gz
!gunzip purchases.txt.gz

--2026-05-03 06:55:25--  http://content.udacity-data.com/courses/ud617/purchases.txt.gz
Resolving content.udacity-data.com (content.udacity-data.com)... 172.64.148.171, 104.18.39.85, 2a06:98c1:3102::ac40:94ab, ...
Connecting to content.udacity-data.com (content.udacity-data.com)|172.64.148.171|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://content.udacity-data.com/courses/ud617/purchases.txt.gz [following]
--2026-05-03 06:55:25--  https://content.udacity-data.com/courses/ud617/purchases.txt.gz
Connecting to content.udacity-data.com (content.udacity-data.com)|172.64.148.171|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 38454568 (37M) [text/plain]
Saving to: ‘purchases.txt.gz’

purchases.txt.gz    100%[===================>]  36.67M  92.6MB/s    in 0.4s    

2026-05-03 06:55:26 (92.6 MB/s) - ‘purchases.txt.gz’ saved [38454568/38454568]



In [ ]:
!head -10 /content/purchases.txt

2012-01-01	09:00	San Jose	Men's Clothing	214.05	Amex
2012-01-01	09:00	Fort Worth	Women's Clothing	153.57	Visa
2012-01-01	09:00	San Diego	Music	66.08	Cash
2012-01-01	09:00	Pittsburgh	Pet Supplies	493.51	Discover
2012-01-01	09:00	Omaha	Children's Clothing	235.63	MasterCard
2012-01-01	09:00	Stockton	Men's Clothing	247.18	MasterCard
2012-01-01	09:00	Austin	Cameras	379.6	Visa
2012-01-01	09:00	New York	Consumer Electronics	296.8	Cash
2012-01-01	09:00	Corpus Christi	Toys	25.38	Discover
2012-01-01	09:00	Fort Worth	Toys	213.88	Visa


1) Instead of breaking the sales down by store, instead retrieve a sales breakdown by product category across all of our stores (instead of by city). Create new mapper and reducer files for this. Hint, only 1 word needs to change

In [ ]:
!touch "mapper1.py"
!touch "reducer1.py"

In [ ]:
%%writefile hdemu.py
import sys
import subprocess

mapper = sys.argv[1]
reducer = sys.argv[2]
input_file = sys.argv[3]

# Run mapper
with open(input_file, 'r') as f:
    mapper_proc = subprocess.Popen(['python', mapper], stdin=f, stdout=subprocess.PIPE, text=True)
    mapper_output = mapper_proc.communicate()[0]

# Sort mapper output (simulate shuffle)
sorted_output = sorted(mapper_output.strip().split("\n"))

# Run reducer
reducer_proc = subprocess.Popen(['python', reducer], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
reducer_output = reducer_proc.communicate("\n".join(sorted_output))[0]

print(reducer_output)

Writing hdemu.py


In [ ]:
!python hdemu.py mapper1.py reducer1.py purchases.txt

Total sales for Toys = 57463477.10999948



2. Find the monetary value for the highest individual sale for each separate store.

In [ ]:
!touch "mapper2.py"
!touch "reducer2.py"

In [ ]:
!python hdemu.py mapper2.py reducer2.py purchases.txt

Highest individual sale for Reno = 499.99



3. Find the total sales value across all the stores, and the total number of sales. Assume there is only one reducer.

In [ ]:
!touch "mapper3.py"
!touch "reducer3.py"

In [ ]:
!python hdemu.py mapper3.py reducer3.py purchases.txt

Number of sales = 4138476, with total value = 1034457953.2599595



4. Write a MapReduce program to find number of hits for each different file on the Web site.

In [ ]:
#ask for dataset
!wget https://raw.githubusercontent.com/databricks/LearningSparkV2/master/databricks-datasets/cs100/lab2/data-001/apache.access.log.PROJECT -O access_log


--2026-05-03 07:37:39--  https://raw.githubusercontent.com/databricks/LearningSparkV2/master/databricks-datasets/cs100/lab2/data-001/apache.access.log.PROJECT
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-03 07:37:39 ERROR 404: Not Found.



In [ ]:
!touch "mapper4.py"
!touch "reducer4.py"

In [ ]:
%%writefile mapper4.py
import sys

target = "/assets/js/the-associates.js"

for line in sys.stdin:
    parts = line.split('"')

    if len(parts) > 1:
        request = parts[1]
        fields = request.split()

        if len(fields) > 1:
            url = fields[1]

            if url == target:
                print("count\t1")

Overwriting mapper4.py


In [ ]:
%%writefile reducer4.py
import sys

count = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    key, value = line.split("\t")
    count += int(value)

print(f"/assets/js/the-associates.js = {count}")

Overwriting reducer4.py


In [ ]:
!python hdemu.py mapper4.py reducer4.py access_log

/assets/js/the-associates.js = 0



5. Write a MapReduce program which determines the number of hits to the site made by each different IP address.

In [ ]:
%%writefile mapper5.py
import sys

target_ip = "10.99.99.186"

for line in sys.stdin:
    ip = line.split()[0]

    if ip == target_ip:
        print("count\t1")

Overwriting mapper5.py


In [ ]:
%%writefile reducer5.py
import sys

count = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    key, value = line.split("\t")
    count += int(value)

print(f"Hits made by IP address 10.99.99.186 = {count}")

Overwriting reducer5.py


In [ ]:
!python hdemu.py mapper5.py reducer5.py access_log

Hits made by IP address 10.99.99.186 = 0



6. Find the most popular file on the website, i.e. whose file path occurs most often in access_log. Your reducer should output the file’s path and number of times it appears in the log

In [ ]:
%%writefile mapper6.py
import sys

for line in sys.stdin:
    parts = line.split('"')

    if len(parts) > 1:
        request = parts[1]
        fields = request.split()

        if len(fields) > 1:
            url = fields[1]
            print(f"{url}\t1")

Overwriting mapper6.py


In [ ]:
%%writefile reducer6.py
import sys

counts = {}

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    key, value = line.split("\t")
    counts[key] = counts.get(key, 0) + int(value)

# check empty case
if not counts:
    print("No data found. Check input file.")
    sys.exit()

# find max
max_file = max(counts, key=counts.get)
max_count = counts[max_file]

file_name = max_file.split("/")[-1]

print(f"Most popular file = {file_name}")
print(f"Number of occurrences = {max_count}")

Overwriting reducer6.py


In [ ]:
!python hdemu.py mapper6.py reducer6.py access_log

Most popular file = product_1
Number of occurrences = 30285



In [ ]:
!head access_log | python mapper6.py

/downloads/product_1	1
/downloads/product_1	1
/downloads/product_1	1
/downloads/product_1	1
/downloads/product_2	1
/downloads/product_1	1
/downloads/product_2	1
/downloads/product_1	1
/downloads/product_1	1
/downloads/product_1	1


In [ ]:
!grep "the-associates.js" access_log | wc -l
!grep "10.99.99.186" access_log | wc -l

0
0


2.  Download the text to Alice's Adventures in Wonderland from
http://www.gutenberg.org/cache/epub/11/pg11.txt and run wordcount on it. How
many times does the word Cheshire occur? (Do not include the word 'Cheshire with
an apostrophe. The string -->'Cheshire<-- does not count)

In [ ]:
import re

with open("/content/pg11.txt", "r", encoding="utf-8") as f:
    text = f.read()

words = re.findall(r"\bCheshire\b", text)

print("Count of 'Cheshire':", len(words))

Count of 'Cheshire': 6


3.  The set of example MapReduce applications includes wordmedian, which
computes the median length of words in a text file. If you run wordmedian using
words.txt on shakespeare.txt file as input, what is the median word length?

In [1]:
!wget https://www.gutenberg.org/files/100/100-0.txt -O shakespeare.txt

--2026-05-04 20:02:44--  https://www.gutenberg.org/files/100/100-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5422721 (5.2M) [text/plain]
Saving to: ‘shakespeare.txt’

shakespeare.txt     100%[===================>]   5.17M  11.0MB/s    in 0.5s    

2026-05-04 20:02:45 (11.0 MB/s) - ‘shakespeare.txt’ saved [5422721/5422721]



In [2]:
import re

with open("/content/shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

words = re.findall(r"\b\w+\b", text)
lengths = sorted(len(w) for w in words)

n = len(lengths)

median = (lengths[n//2] if n % 2 != 0
          else (lengths[n//2 - 1] + lengths[n//2]) / 2)

print("Median word length:", median)

Median word length: 4.0
